In [1]:
# !pip install catboost xgboost lightgbm -q

In [2]:
import os
import joblib
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (

    accuracy_score,
    classification_report,
    confusion_matrix,

    roc_auc_score,
    average_precision_score,

    precision_score,
    recall_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    f1_score

)
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
MODEL_PATH = (
"/content/drive/MyDrive/AI_MANUAL_PROJECT/models/"
)

os.makedirs(
    MODEL_PATH,
    exist_ok=True
)

In [4]:
df = pd.read_csv(

"/content/drive/MyDrive/AI_MANUAL_PROJECT/datasets/manual_finbert_dataset.csv"

)

df["time"] = pd.to_datetime(df["time"])

df = df.sort_values(
    "time"
).reset_index(drop=True)

print(df.shape)

(80232, 56)


In [5]:
encoders = {}

for col in df.columns:

    if df[col].dtype=="object":

        le = LabelEncoder()

        df[col] = le.fit_transform(
            df[col].astype(str)
        )

        encoders[col]=le

In [6]:
FEATURES = [

"EMA20",
"EMA50",
"RSI",
"ATR",
"NATR",

"BB_WIDTH",
"CHAIKIN_VOL",
"VQI",

"TREND_STRENGTH",
"CHANNEL_POSITION",

"SLOPE_SIGNAL",
"POWER_SCORE",

"ACTIVE_PASSIVE",

"FINANCIAL_STRENGTH",

"pair",
"timeframe",

"MARKET_STATE",

"CANDLE_PATTERN",

"FREQUENCY_TYPE",

"h1_positive",
"h1_negative",
"h1_neutral",

"h2_positive",
"h2_negative",
"h2_neutral",

"h3_positive",
"h3_negative",
"h3_neutral",

"overall_positive",
"overall_negative",
"overall_neutral",

"news_strength",

"dominant_sentiment"

]

In [7]:
TARGET = "TARGET"

df.dropna(
    inplace=True
)

train_end = int(
    len(df)*0.70
)

val_end = int(
    len(df)*0.85
)

train_df = df.iloc[:train_end]

val_df = df.iloc[
    train_end:val_end
]

test_df = df.iloc[
    val_end:
]

X_train = train_df[
    FEATURES
]

y_train = train_df[
    TARGET
]

X_val = val_df[
    FEATURES
]

y_val = val_df[
    TARGET
]

X_test = test_df[
    FEATURES
]

y_test = test_df[
    TARGET
]

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

print()

print(y_train.value_counts())

print()

print(y_val.value_counts())

print()

print(y_test.value_counts())

(56162, 33)
(12035, 33)
(12035, 33)

TARGET
0    29747
1    26415
Name: count, dtype: int64

TARGET
0    6795
1    5240
Name: count, dtype: int64

TARGET
0    7020
1    5015
Name: count, dtype: int64


In [8]:
cat_model = CatBoostClassifier(

    iterations=5000,

    learning_rate=0.02,

    depth=8,

    l2_leaf_reg=5,

    bagging_temperature=1,

    random_strength=1,

    loss_function='Logloss',

    eval_metric='PRAUC',

    auto_class_weights='Balanced',

    random_seed=42,

    verbose=100,

    od_type='Iter',

    od_wait=200

)

cat_model.fit(

    X_train,
    y_train,

    eval_set=(X_val,y_val),

    use_best_model=True

)

0:	learn: 0.6949861	test: 0.6549171	best: 0.6549171 (0)	total: 326ms	remaining: 27m 9s
100:	learn: 0.7187892	test: 0.6618432	best: 0.6618432 (100)	total: 15.1s	remaining: 12m 14s
200:	learn: 0.7322798	test: 0.6635364	best: 0.6635364 (200)	total: 23.8s	remaining: 9m 29s
300:	learn: 0.7435736	test: 0.6644329	best: 0.6644329 (300)	total: 29.3s	remaining: 7m 37s
400:	learn: 0.7534478	test: 0.6648045	best: 0.6648392 (364)	total: 36.7s	remaining: 7m
500:	learn: 0.7624827	test: 0.6651877	best: 0.6652519 (496)	total: 42.1s	remaining: 6m 18s
600:	learn: 0.7705474	test: 0.6652521	best: 0.6654273 (536)	total: 48.3s	remaining: 5m 53s
700:	learn: 0.7792975	test: 0.6652394	best: 0.6654273 (536)	total: 54.6s	remaining: 5m 35s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.6654273493
bestIteration = 536

Shrink model to first 537 iterations.


CatBoostClassifier(auto_class_weights='Balanced', bagging_temperature=1, depth=8, eval_metric='PRAUC', iterations=5000, l2_leaf_reg=5, learning_rate=0.02, loss_function='Logloss', od_type='Iter', od_wait=200, random_seed=42, random_strength=1, verbose=100)

In [9]:
train_pred = cat_model.predict(X_train)

val_pred = cat_model.predict(X_val)

test_pred = cat_model.predict(X_test)

train_pred = train_pred.astype(int)

val_pred = val_pred.astype(int)

test_pred = test_pred.astype(int)

train_probs = cat_model.predict_proba(X_train)[:,1]

val_probs = cat_model.predict_proba(X_val)[:,1]

test_probs = cat_model.predict_proba(X_test)[:,1]

In [10]:
print("\n========== ACCURACY ==========")

print(
"Train Accuracy:",
round(
accuracy_score(
y_train,
train_pred
),4)
)

print(
"Validation Accuracy:",
round(
accuracy_score(
y_val,
val_pred
),4)
)

print(
"Test Accuracy:",
round(
accuracy_score(
y_test,
test_pred
),4)
)


========== ACCURACY ==========
Train Accuracy: 0.696
Validation Accuracy: 0.6587
Test Accuracy: 0.6442


In [11]:
print("\n========== TRAIN REPORT ==========")

print(
classification_report(
y_train,
train_pred
)
)

print("\n========== VALIDATION REPORT ==========")

print(
classification_report(
y_val,
val_pred
)
)

print("\n========== TEST REPORT ==========")

print(
classification_report(
y_test,
test_pred
)
)


========== TRAIN REPORT ==========
              precision    recall  f1-score   support

           0       0.73      0.68      0.70     29747
           1       0.66      0.72      0.69     26415

    accuracy                           0.70     56162
   macro avg       0.70      0.70      0.70     56162
weighted avg       0.70      0.70      0.70     56162


========== VALIDATION REPORT ==========
              precision    recall  f1-score   support

           0       0.70      0.69      0.70      6795
           1       0.61      0.62      0.61      5240

    accuracy                           0.66     12035
   macro avg       0.65      0.65      0.65     12035
weighted avg       0.66      0.66      0.66     12035


========== TEST REPORT ==========
              precision    recall  f1-score   support

           0       0.69      0.71      0.70      7020
           1       0.58      0.55      0.56      5015

    accuracy                           0.64     12035
   macro avg    

In [12]:
print("\nTRAIN CONFUSION MATRIX")

print(
confusion_matrix(
y_train,
train_pred
)
)

print("\nVALIDATION CONFUSION MATRIX")

print(
confusion_matrix(
y_val,
val_pred
)
)

print("\nTEST CONFUSION MATRIX")

print(
confusion_matrix(
y_test,
test_pred
)
)


TRAIN CONFUSION MATRIX
[[20173  9574]
 [ 7499 18916]]

VALIDATION CONFUSION MATRIX
[[4689 2106]
 [2001 3239]]

TEST CONFUSION MATRIX
[[4983 2037]
 [2245 2770]]


In [13]:
print(

"\nROC-AUC =",

round(
roc_auc_score(
y_test,
test_probs
),4)

)

print(

"PR-AUC =",

round(
average_precision_score(
y_test,
test_probs
),4)

)

print(

"Precision =",

round(
precision_score(
y_test,
test_pred
),4)

)

print(

"Recall =",

round(
recall_score(
y_test,
test_pred
),4)

)

print(

"Balanced Accuracy =",

round(
balanced_accuracy_score(
y_test,
test_pred
),4)

)

print(

"MCC =",

round(
matthews_corrcoef(
y_test,
test_pred
),4)

)


ROC-AUC = 0.6927
PR-AUC = 0.5999
Precision = 0.5762
Recall = 0.5523
Balanced Accuracy = 0.6311
MCC = 0.2639


In [14]:
importance = pd.DataFrame({

"Feature":FEATURES,

"Importance":
cat_model.feature_importances_

})

importance = importance.sort_values(

by="Importance",

ascending=False

)

print(
importance
)

               Feature  Importance
4                 NATR   24.562706
11         POWER_SCORE    4.847233
6          CHAIKIN_VOL    4.669796
9     CHANNEL_POSITION    4.457719
7                  VQI    4.238631
2                  RSI    3.318804
20         h1_negative    3.133310
10        SLOPE_SIGNAL    2.819384
1                EMA50    2.739274
0                EMA20    2.645490
23         h2_negative    2.417186
14                pair    2.315503
27          h3_neutral    2.260547
22         h2_positive    2.245989
8       TREND_STRENGTH    2.220635
26         h3_negative    2.215343
24          h2_neutral    2.196775
21          h1_neutral    2.164657
25         h3_positive    2.073633
3                  ATR    2.007028
30     overall_neutral    2.006468
19         h1_positive    1.978288
28    overall_positive    1.943358
31       news_strength    1.896181
29    overall_negative    1.869276
5             BB_WIDTH    1.853501
13  FINANCIAL_STRENGTH    1.752058
15           timefra

In [15]:
best_score = 0

for threshold in np.arange(0.35,0.71,0.01):

    preds = (

        test_probs > threshold

    ).astype(int)

    precision = precision_score(
        y_test,
        preds
    )

    recall = recall_score(
        y_test,
        preds
    )

    f1 = f1_score(
        y_test,
        preds
    )

    score = (

        0.5*f1

        +

        0.3*precision

        +

        0.2*recall

    )

    if score > best_score:

        best_score = score

        best_threshold = threshold

        best_f1 = f1

        best_precision = precision

        best_recall = recall

In [16]:
print(

"BEST THRESHOLD =",

round(
best_threshold,
2)

)

print(

"BEST F1 =",

round(
best_f1,
4)

)

print(

"BEST PRECISION =",

round(
best_precision,
4)

)

print(

"BEST RECALL =",

round(
best_recall,
4)

)

BEST THRESHOLD = 0.35
BEST F1 = 0.6256
BEST PRECISION = 0.5024
BEST RECALL = 0.8291


In [17]:
final_pred = (

test_probs > best_threshold

).astype(int)

print(

classification_report(

y_test,

final_pred

)

)

print(

confusion_matrix(

y_test,

final_pred

)

)

              precision    recall  f1-score   support

           0       0.77      0.41      0.54      7020
           1       0.50      0.83      0.63      5015

    accuracy                           0.59     12035
   macro avg       0.64      0.62      0.58     12035
weighted avg       0.66      0.59      0.57     12035

[[2901 4119]
 [ 857 4158]]


In [18]:
joblib.dump(

cat_model,

MODEL_PATH+

"manual_catboost.pkl"

)

joblib.dump(

best_threshold,

MODEL_PATH+

"catboost_threshold.pkl"

)

print(

"CatBoost saved"

)

CatBoost saved


In [19]:
# ==========================================
# CLASS IMBALANCE
# ==========================================

scale_pos_weight = (

    (y_train == 0).sum()

    /

    (y_train == 1).sum()

)

print(
    "Scale Pos Weight =",
    round(scale_pos_weight,4)
)

Scale Pos Weight = 1.1261


In [20]:
xgb_model = XGBClassifier(

    n_estimators=3000,

    learning_rate=0.02,

    max_depth=5,

    min_child_weight=10,

    subsample=0.8,

    colsample_bytree=0.8,

    gamma=0.2,

    reg_alpha=0.5,

    reg_lambda=3,

    scale_pos_weight=scale_pos_weight,

    eval_metric="aucpr",

    early_stopping_rounds=100,

    tree_method="hist",

    random_state=42,

    n_jobs=-1
)

In [21]:
xgb_model.fit(

    X_train,
    y_train,

    eval_set=[(X_val,y_val)],

    verbose=100

)

train_pred = xgb_model.predict(X_train)

val_pred = xgb_model.predict(X_val)

test_pred = xgb_model.predict(X_test)

train_pred = train_pred.astype(int)

val_pred = val_pred.astype(int)

test_pred = test_pred.astype(int)

train_probs = xgb_model.predict_proba(X_train)[:,1]

val_probs = xgb_model.predict_proba(X_val)[:,1]

test_probs = xgb_model.predict_proba(X_test)[:,1]

[0]	validation_0-aucpr:0.62380
[100]	validation_0-aucpr:0.63415
[200]	validation_0-aucpr:0.63710
[300]	validation_0-aucpr:0.63791
[400]	validation_0-aucpr:0.63827
[491]	validation_0-aucpr:0.63790


In [22]:
print("\n========== ACCURACY ==========")

print(
"Train Accuracy:",
round(
accuracy_score(
y_train,
train_pred
),4)
)

print(
"Validation Accuracy:",
round(
accuracy_score(
y_val,
val_pred
),4)
)

print(
"Test Accuracy:",
round(
accuracy_score(
y_test,
test_pred
),4)
)


========== ACCURACY ==========
Train Accuracy: 0.6854
Validation Accuracy: 0.6595
Test Accuracy: 0.646


In [23]:
print("\n========== TRAIN REPORT ==========")

print(
classification_report(
y_train,
train_pred
)
)

print("\n========== VALIDATION REPORT ==========")

print(
classification_report(
y_val,
val_pred
)
)

print("\n========== TEST REPORT ==========")

print(
classification_report(
y_test,
test_pred
)
)


========== TRAIN REPORT ==========
              precision    recall  f1-score   support

           0       0.72      0.67      0.69     29747
           1       0.65      0.70      0.68     26415

    accuracy                           0.69     56162
   macro avg       0.69      0.69      0.69     56162
weighted avg       0.69      0.69      0.69     56162


========== VALIDATION REPORT ==========
              precision    recall  f1-score   support

           0       0.70      0.70      0.70      6795
           1       0.61      0.61      0.61      5240

    accuracy                           0.66     12035
   macro avg       0.65      0.65      0.65     12035
weighted avg       0.66      0.66      0.66     12035


========== TEST REPORT ==========
              precision    recall  f1-score   support

           0       0.69      0.72      0.70      7020
           1       0.58      0.55      0.56      5015

    accuracy                           0.65     12035
   macro avg    

In [24]:
print("\nTRAIN CONFUSION MATRIX")

print(
confusion_matrix(
y_train,
train_pred
)
)

print("\nVALIDATION CONFUSION MATRIX")

print(
confusion_matrix(
y_val,
val_pred
)
)

print("\nTEST CONFUSION MATRIX")

print(
confusion_matrix(
y_test,
test_pred
)
)


TRAIN CONFUSION MATRIX
[[19912  9835]
 [ 7835 18580]]

VALIDATION CONFUSION MATRIX
[[4731 2064]
 [2034 3206]]

TEST CONFUSION MATRIX
[[5036 1984]
 [2276 2739]]


In [25]:
print(

"\nROC-AUC =",

round(
roc_auc_score(
y_test,
test_probs
),4)

)

print(

"PR-AUC =",

round(
average_precision_score(
y_test,
test_probs
),4)

)

print(

"Precision =",

round(
precision_score(
y_test,
test_pred
),4)

)

print(

"Recall =",

round(
recall_score(
y_test,
test_pred
),4)

)

print(

"Balanced Accuracy =",

round(
balanced_accuracy_score(
y_test,
test_pred
),4)

)

print(

"MCC =",

round(
matthews_corrcoef(
y_test,
test_pred
),4)

)


ROC-AUC = 0.6923
PR-AUC = 0.5995
Precision = 0.5799
Recall = 0.5462
Balanced Accuracy = 0.6318
MCC = 0.2661


In [26]:
importance = pd.DataFrame({

"Feature":FEATURES,

"Importance":
xgb_model.feature_importances_

})

importance = importance.sort_values(

by="Importance",

ascending=False

)

print(
importance
)

               Feature  Importance
4                 NATR    0.217041
0                EMA20    0.091248
18      FREQUENCY_TYPE    0.075404
14                pair    0.066389
15           timeframe    0.048625
13  FINANCIAL_STRENGTH    0.044429
1                EMA50    0.035510
17      CANDLE_PATTERN    0.032868
3                  ATR    0.032326
5             BB_WIDTH    0.024778
7                  VQI    0.019884
11         POWER_SCORE    0.019384
12      ACTIVE_PASSIVE    0.019280
6          CHAIKIN_VOL    0.017946
9     CHANNEL_POSITION    0.015634
8       TREND_STRENGTH    0.015447
31       news_strength    0.014375
28    overall_positive    0.014032
30     overall_neutral    0.013872
2                  RSI    0.013764
29    overall_negative    0.013581
24          h2_neutral    0.013455
20         h1_negative    0.013390
22         h2_positive    0.013375
32  dominant_sentiment    0.013242
10        SLOPE_SIGNAL    0.013111
19         h1_positive    0.013002
23         h2_negati

In [27]:
best_score = 0

for threshold in np.arange(0.20,0.81,0.01):

    preds = (

        test_probs > threshold

    ).astype(int)

    precision = precision_score(
        y_test,
        preds
    )

    recall = recall_score(
        y_test,
        preds
    )

    f1 = f1_score(
        y_test,
        preds
    )

    score = (

        0.6*f1

        +

        0.2*precision

        +

        0.2*recall

    )

    if score > best_score:

        best_score = score

        best_threshold = threshold

        best_f1 = f1

        best_precision = precision

        best_recall = recall

In [28]:
print(

"BEST THRESHOLD =",

round(
best_threshold,
2)

)

print(

"BEST F1 =",

round(
best_f1,
4)

)

print(

"BEST PRECISION =",

round(
best_precision,
4)

)

print(

"BEST RECALL =",

round(
best_recall,
4)

)

BEST THRESHOLD = 0.27
BEST F1 = 0.6247
BEST PRECISION = 0.4708
BEST RECALL = 0.9282


In [29]:
final_pred = (

test_probs > best_threshold

).astype(int)

print(

classification_report(

y_test,

final_pred

)

)

print(

confusion_matrix(

y_test,

final_pred

)

)

              precision    recall  f1-score   support

           0       0.83      0.25      0.39      7020
           1       0.47      0.93      0.62      5015

    accuracy                           0.54     12035
   macro avg       0.65      0.59      0.51     12035
weighted avg       0.68      0.54      0.49     12035

[[1788 5232]
 [ 360 4655]]


In [30]:
joblib.dump(

xgb_model,

MODEL_PATH+

"manual_xgboost.pkl"

)

joblib.dump(

best_threshold,

MODEL_PATH+

"xgboost_threshold.pkl"

)

print(

"XGBoost saved"

)

XGBoost saved


In [31]:
lgb_model = LGBMClassifier(

    n_estimators=1000,

    learning_rate=0.03,

    num_leaves=31,

    max_depth=5,

    min_child_samples=100,

    min_split_gain=0.01,

    subsample=0.8,

    colsample_bytree=0.8,

    reg_alpha=1,

    reg_lambda=5,

    class_weight="balanced",

    random_state=42,

    n_jobs=-1,

    verbose=-1

)

In [32]:
lgb_model.fit(

    X_train,

    y_train,

    eval_set=[(X_val,y_val)]

)
train_pred = lgb_model.predict(X_train)

val_pred = lgb_model.predict(X_val)

test_pred = lgb_model.predict(X_test)

train_pred = train_pred.astype(int)

val_pred = val_pred.astype(int)

test_pred = test_pred.astype(int)

train_probs = lgb_model.predict_proba(X_train)[:,1]

val_probs = lgb_model.predict_proba(X_val)[:,1]

test_probs = lgb_model.predict_proba(X_test)[:,1]

In [33]:
print("\n========== ACCURACY ==========")

print(
"Train Accuracy:",
round(
accuracy_score(
y_train,
train_pred
),4)
)

print(
"Validation Accuracy:",
round(
accuracy_score(
y_val,
val_pred
),4)
)

print(
"Test Accuracy:",
round(
accuracy_score(
y_test,
test_pred
),4)
)


========== ACCURACY ==========
Train Accuracy: 0.7137
Validation Accuracy: 0.6539
Test Accuracy: 0.6425


In [34]:
print("\n========== TRAIN REPORT ==========")

print(
classification_report(
y_train,
train_pred
)
)

print("\n========== VALIDATION REPORT ==========")

print(
classification_report(
y_val,
val_pred
)
)

print("\n========== TEST REPORT ==========")

print(
classification_report(
y_test,
test_pred
)
)


========== TRAIN REPORT ==========
              precision    recall  f1-score   support

           0       0.75      0.70      0.72     29747
           1       0.68      0.73      0.71     26415

    accuracy                           0.71     56162
   macro avg       0.71      0.71      0.71     56162
weighted avg       0.72      0.71      0.71     56162


========== VALIDATION REPORT ==========
              precision    recall  f1-score   support

           0       0.70      0.68      0.69      6795
           1       0.60      0.62      0.61      5240

    accuracy                           0.65     12035
   macro avg       0.65      0.65      0.65     12035
weighted avg       0.66      0.65      0.65     12035


========== TEST REPORT ==========
              precision    recall  f1-score   support

           0       0.69      0.71      0.70      7020
           1       0.57      0.55      0.56      5015

    accuracy                           0.64     12035
   macro avg    

In [35]:
print("\nTRAIN CONFUSION MATRIX")

print(
confusion_matrix(
y_train,
train_pred
)
)

print("\nVALIDATION CONFUSION MATRIX")

print(
confusion_matrix(
y_val,
val_pred
)
)

print("\nTEST CONFUSION MATRIX")

print(
confusion_matrix(
y_test,
test_pred
)
)


TRAIN CONFUSION MATRIX
[[20685  9062]
 [ 7018 19397]]

VALIDATION CONFUSION MATRIX
[[4635 2160]
 [2005 3235]]

TEST CONFUSION MATRIX
[[4975 2045]
 [2258 2757]]


In [36]:
print(

"\nROC-AUC =",

round(
roc_auc_score(
y_test,
test_probs
),4)

)

print(

"PR-AUC =",

round(
average_precision_score(
y_test,
test_probs
),4)

)

print(

"Precision =",

round(
precision_score(
y_test,
test_pred
),4)

)

print(

"Recall =",

round(
recall_score(
y_test,
test_pred
),4)

)

print(

"Balanced Accuracy =",

round(
balanced_accuracy_score(
y_test,
test_pred
),4)

)

print(

"MCC =",

round(
matthews_corrcoef(
y_test,
test_pred
),4)

)


ROC-AUC = 0.691
PR-AUC = 0.5993
Precision = 0.5741
Recall = 0.5498
Balanced Accuracy = 0.6292
MCC = 0.2602


In [37]:
importance = pd.DataFrame({

"Feature":FEATURES,

"Importance":
lgb_model.feature_importances_

})

importance = importance.sort_values(

by="Importance",

ascending=False

)

print(
importance
)

               Feature  Importance
4                 NATR        1397
9     CHANNEL_POSITION        1100
11         POWER_SCORE        1049
6          CHAIKIN_VOL        1043
2                  RSI        1013
7                  VQI         974
10        SLOPE_SIGNAL         827
27          h3_neutral         777
21          h1_neutral         733
5             BB_WIDTH         713
19         h1_positive         710
20         h1_negative         697
26         h3_negative         693
0                EMA20         682
24          h2_neutral         669
13  FINANCIAL_STRENGTH         661
8       TREND_STRENGTH         659
3                  ATR         648
23         h2_negative         644
30     overall_neutral         628
1                EMA50         623
31       news_strength         608
28    overall_positive         608
29    overall_negative         606
25         h3_positive         583
22         h2_positive         570
15           timeframe         156
18      FREQUENCY_TY

In [38]:
best_score = 0

for threshold in np.arange(0.20,0.81,0.01):

    preds = (

        test_probs > threshold

    ).astype(int)

    precision = precision_score(
        y_test,
        preds
    )

    recall = recall_score(
        y_test,
        preds
    )

    f1 = f1_score(
        y_test,
        preds
    )

    score = (

        0.6*f1

        +

        0.2*precision

        +

        0.2*recall

    )

    if score > best_score:

        best_score = score

        best_threshold = threshold

        best_f1 = f1

        best_precision = precision

        best_recall = recall

In [39]:
print(

"BEST THRESHOLD =",

round(
best_threshold,
2)

)

print(

"BEST F1 =",

round(
best_f1,
4)

)

print(

"BEST PRECISION =",

round(
best_precision,
4)

)

print(

"BEST RECALL =",

round(
best_recall,
4)

)

BEST THRESHOLD = 0.24
BEST F1 = 0.6207
BEST PRECISION = 0.4628
BEST RECALL = 0.9426


In [40]:
final_pred = (

test_probs > best_threshold

).astype(int)

print(

classification_report(

y_test,

final_pred

)

)

print(

confusion_matrix(

y_test,

final_pred

)

)

              precision    recall  f1-score   support

           0       0.84      0.22      0.35      7020
           1       0.46      0.94      0.62      5015

    accuracy                           0.52     12035
   macro avg       0.65      0.58      0.48     12035
weighted avg       0.68      0.52      0.46     12035

[[1532 5488]
 [ 288 4727]]


In [41]:
joblib.dump(

lgb_model,

MODEL_PATH+

"manual_lgbm.pkl"

)

joblib.dump(

best_threshold,

MODEL_PATH+

"lgbm_threshold.pkl"

)

print(

"lgbm saved"

)

lgbm saved


In [42]:
cat_probs = cat_model.predict_proba(X_test)[:,1]

xgb_probs = xgb_model.predict_proba(X_test)[:,1]

lgb_probs = lgb_model.predict_proba(X_test)[:,1]

In [43]:
ensemble_probs = (

      0.40*cat_probs

    + 0.35*xgb_probs

    + 0.25*lgb_probs

)

In [44]:
best_score = 0

for threshold in np.arange(0.40,0.70,0.01):

    preds = (ensemble_probs > threshold).astype(int)

    precision = precision_score(y_test,preds)

    recall = recall_score(y_test,preds)

    f1 = f1_score(y_test,preds)

    score = (
        0.4*f1
        +
        0.4*precision
        +
        0.2*recall
    )

    if score > best_score:

        best_score = score

        best_threshold = threshold

        best_precision = precision

        best_recall = recall

        best_f1 = f1

In [45]:
print("BEST THRESHOLD =",round(best_threshold,2))

print("BEST F1 =",round(best_f1,4))

print("BEST PRECISION =",round(best_precision,4))

print("BEST RECALL =",round(best_recall,4))

BEST THRESHOLD = 0.4
BEST F1 = 0.6177
BEST PRECISION = 0.525
BEST RECALL = 0.7501


In [46]:
final_pred = (

ensemble_probs > best_threshold

).astype(int)

In [47]:
print(classification_report(

    y_test,

    final_pred

))

print()

print(confusion_matrix(

    y_test,

    final_pred

))

print()

print(

"Accuracy =",

round(

accuracy_score(

y_test,

final_pred

),

4

)

)

print(

"Precision =",

round(

precision_score(

y_test,

final_pred

),

4

)

)

print(

"Recall =",

round(

recall_score(

y_test,

final_pred

),

4

)

)

print(

"F1 =",

round(

f1_score(

y_test,

final_pred

),

4

)

)

print(

"ROC-AUC =",

round(

roc_auc_score(

y_test,

ensemble_probs

),

4

)

)

print(

"PR-AUC =",

round(

average_precision_score(

y_test,

ensemble_probs

),

4

)

)

print(

"Balanced Accuracy =",

round(

balanced_accuracy_score(

y_test,

final_pred

),

4

)

)

print(

"MCC =",

round(

matthews_corrcoef(

y_test,

final_pred

),

4

)

)

              precision    recall  f1-score   support

           0       0.74      0.52      0.61      7020
           1       0.52      0.75      0.62      5015

    accuracy                           0.61     12035
   macro avg       0.63      0.63      0.61     12035
weighted avg       0.65      0.61      0.61     12035


[[3616 3404]
 [1253 3762]]

Accuracy = 0.613
Precision = 0.525
Recall = 0.7501
F1 = 0.6177
ROC-AUC = 0.6928
PR-AUC = 0.6004
Balanced Accuracy = 0.6326
MCC = 0.2664


In [48]:
joblib.dump(

best_threshold,

MODEL_PATH+"ensemble_threshold.pkl"

)

joblib.dump(

encoders,

MODEL_PATH+"manual_label_encoders.pkl"

)

print("ALL MODELS SAVED")

ALL MODELS SAVED
